# 04 · Dataset generation (laptop → H100 node), format, counterfactual branching

`scripts/generate_dataset.py` runs one MuJoCo env per worker process. Episodes are independent and
seeded, so throughput scales with CPU cores, and a big run can be sharded across nodes with
`--seed0`. On a headless GPU node, rendering goes through EGL (`MUJOCO_GL=egl`).

Each episode folder uses the **same layout as the real WR1 data** read by
`GaugeFailClassification/src/data/complex_datasets/trajectory.py`: `episode.parquet` with an int64-ns
`timestamp`, `observation.state` and `action`, one `<camera>.mp4` per feed, plus `annotations.json`
and `feature_report.json`. On top of that there are `priv.*` ground-truth columns and
`sim_states.npz` (full simulator snapshot per step).

In [1]:
# Setup cell, inlined into every notebook by notebooks/build.py.
# Locally: finds the repo. On a Colab runtime (also VS Code connected to Colab), where the code is not
# on the VM: fetches it from (1) PLUGSIM_REPO (git URL, default: the public repo), (2) /content/RobotPluggingSim.zip, or
# (3) MyDrive/RobotPluggingSim (Drive), installs it, and keeps data/ + runs/ on Drive when mounted.
import os, sys, pathlib, subprocess
DEFAULT_REPO = "https://github.com/carlo-scr/RobotPluggingSim.git"
PLUGSIM_REPO = os.environ.get("PLUGSIM_REPO") or DEFAULT_REPO  # Colab: code is cloned from here
USE_DRIVE = os.environ.get("PLUGSIM_DRIVE", "1") == "1"  # set False if drive.mount hangs in your client
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    # numpy's BLAS: one thread (the sim solves tiny systems; extra threads only add contention)
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
    os.environ.setdefault("MKL_NUM_THREADS", "1")
    os.environ.setdefault("MUJOCO_GL", "egl")
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    CODE = pathlib.Path("/content/RobotPluggingSim")
    DRIVE = pathlib.Path("/content/drive/MyDrive")
    if (CODE / ".git").exists():
        subprocess.run(["git", "-C", str(CODE), "pull", "-q"])  # pick up pushed changes
    if not (CODE / "plugsim").exists():
        if PLUGSIM_REPO:
            subprocess.run(["git", "clone", "-q", PLUGSIM_REPO, str(CODE)], check=True)
        elif pathlib.Path("/content/RobotPluggingSim.zip").exists():
            subprocess.run(["unzip", "-q", "-o", "/content/RobotPluggingSim.zip", "-d", "/content"], check=True)
    if USE_DRIVE and not DRIVE.exists():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:  # e.g. not supported by the client; fine with PLUGSIM_REPO or the zip
            print("Drive not mounted:", e)
    if not (CODE / "plugsim").exists():
        if (DRIVE / "RobotPluggingSim").exists():
            subprocess.run(["rsync", "-a", "--exclude", ".venv", "--exclude", "data", "--exclude", "runs", "--exclude", "media",
                            f"{DRIVE}/RobotPluggingSim/", f"{CODE}/"], check=True)
        else:  # last resort: the public repo
            subprocess.run(["git", "clone", "-q", DEFAULT_REPO, str(CODE)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE)], check=True)
    if DRIVE.exists():  # persist datasets and checkpoints across sessions / notebooks
        for d in ("data", "runs"):
            tgt = DRIVE / "plugsim_runs" / d
            tgt.mkdir(parents=True, exist_ok=True)
            link = CODE / d
            if not link.is_symlink():
                subprocess.run(["rm", "-rf", str(link)])
                link.symlink_to(tgt)
    else:
        print("WARNING: no Drive: data/ and runs/ live on the VM and are lost when the runtime stops")
    try:
        import torch
        if torch.cuda.is_available():
            os.environ.setdefault("PLUGSIM_SCALE", "a100")
    except ImportError:
        pass
    os.chdir(CODE)
ROOT = pathlib.Path.cwd()
while not (ROOT / "plugsim").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
    os.environ.setdefault("MUJOCO_GL", "egl")  # headless GPU nodes
print("ROOT:", ROOT, "| colab:", IN_COLAB, "| scale:", os.environ.get("PLUGSIM_SCALE", "local"))

ROOT: . | colab: False | scale: local


In [2]:
import os, sys, pathlib, subprocess, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

OUT = ROOT / "data" / "smoke"   # small local dataset (gitignored)
N, WORKERS = 12, 4

## Generate a small dataset locally

In [3]:
t = time.time()
cmd = [sys.executable, str(ROOT / "scripts" / "generate_dataset.py"), "--out", str(OUT), "--n", str(N),
       "--workers", str(WORKERS), "--mix", "nominal:0.5,known_failure:0.35,ood:0.15"]
r = subprocess.run(cmd, capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines() if "WARNING" not in l)[-1500:]); print(r.stderr[-500:] if r.returncode else "")
wall = time.time() - t
idx = pd.read_parquet(OUT / "index_0.parquet")
size_mb = sum(f.stat().st_size for f in OUT.rglob("*") if f.is_file()) / 1e6
print(f"{N} episodes, {wall:.0f}s wall with {WORKERS} workers, {size_mb / N:.1f} MB/episode")
idx[["episode", "category", "outcome", "failure_mode", "injected", "severity", "ood", "retries", "duration_s"]]

[1/12] 17s elapsed, 0.06 ep/s, eta 191s
[2/12] 19s elapsed, 0.10 ep/s, eta 95s
[3/12] 19s elapsed, 0.15 ep/s, eta 58s
[4/12] 26s elapsed, 0.16 ep/s, eta 51s
[5/12] 34s elapsed, 0.15 ep/s, eta 47s
[6/12] 34s elapsed, 0.18 ep/s, eta 34s
[7/12] 38s elapsed, 0.18 ep/s, eta 27s
[8/12] 44s elapsed, 0.18 ep/s, eta 22s
[9/12] 49s elapsed, 0.18 ep/s, eta 16s
[10/12] 54s elapsed, 0.19 ep/s, eta 11s
[11/12] 54s elapsed, 0.20 ep/s, eta 5s
[12/12] 61s elapsed, 0.20 ep/s, eta 0s
category       outcome
known_failure  failure    3
               success    1
nominal        success    5
ood            success    3
done: 12 episodes, 0 errors, 61s -> ./data/smoke

12 episodes, 62s wall with 4 workers, 9.4 MB/episode


,episode,category,outcome,failure_mode,injected,severity,ood,retries,duration_s
0,ep_0000000,known_failure,failure,not_seated,partial_insert,0.040974,,0,8.05
1,ep_0000001,nominal,success,none,NaN,NaN,,0,9.00
3,ep_0000002,known_failure,failure,grasp_missed,weak_grip,0.912756,,0,12.65
2,ep_0000003,known_failure,success,none,bump,0.543625,,0,9.15
5,ep_0000004,ood,success,none,NaN,NaN,boot_color+stiff_cable,0,8.15
6,ep_0000005,ood,success,none,NaN,NaN,camera_noise,0,8.60
4,ep_0000006,known_failure,failure,dropped,drop,0.175656,,0,6.90
7,ep_0000007,ood,success,none,NaN,NaN,camera_shift,0,8.90
8,ep_0000008,nominal,success,none,NaN,NaN,,0,7.40
9,ep_0000009,nominal,success,none,NaN,NaN,,0,9.70


## What's in an episode

In [4]:
from plugsim.datagen import load_episode
ep = load_episode(idx.path.iloc[0], load_frames=True)
df = ep["df"]
print("columns:", list(df.columns))
print("state dims:", len(ep["features"]["observation_state_names"]), "| action dims:", len(ep["features"]["action_names"]))
print("cameras:", ep["features"]["cameras"], "| frames:", {c: v.shape for c, v in ep["frames"].items()})
print(json.dumps({k: ep["annotations"][k] for k in ("category", "outcome", "failure_mode", "injected_failure", "target_port", "latch_time_s", "drop_times_s")}, indent=1, default=str))
df.head(3)

columns: ['timestamp', 'frame_index', 'observation.state', 'action', 'action.ee_target', 'action.expert', 'priv.phase', 'priv.plug_pos', 'priv.plug_quat', 'priv.tip_pos', 'priv.insert_depth', 'priv.lateral_err', 'priv.angle_err', 'priv.latched_port', 'priv.grasped_by']
state dims: 44 | action dims: 16
cameras: ['right_gripper_front', 'right_gripper_rear', 'left_gripper_front', 'left_gripper_rear'] | frames: {'right_gripper_front': (161, 216, 384, 3), 'right_gripper_rear': (161, 216, 384, 3), 'left_gripper_front': (161, 216, 384, 3), 'left_gripper_rear': (161, 216, 384, 3)}
{
 "category": "known_failure",
 "outcome": "failure",
 "failure_mode": "not_seated",
 "injected_failure": {
  "kind": "partial_insert",
  "severity": 0.04097352393619469,
  "params": {
   "depth_frac": 0.8254158856382832
  }
 },
 "target_port": 38,
 "latch_time_s": null,
 "drop_times_s": []
}


,timestamp,frame_index,observation.state,action,action.ee_target,action.expert,priv.phase,priv.plug_pos,priv.plug_quat,priv.tip_pos,priv.insert_depth,priv.lateral_err,priv.angle_err,priv.latched_port,priv.grasped_by
0,0,0,"[0.019520996, -0.47598535, -0.009390943, -2.77...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.499...","[0.3566572, -0.25636595, 1.0521638, 0.00047673...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.499...",approach,"[0.39746016, 0.20965382, 0.8037997]","[0.9996047, -0.00017272736, -0.006196478, -0.0...","[0.43440166, 0.20762534, 0.80425835]",-0.092784,0.238086,0.058743,-1,-1
1,50000000,1,"[0.01914619, -0.47598028, -0.008985005, -2.770...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.000...","[0.3566572, -0.25636595, 1.0521638, 0.00047673...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.000...",approach,"[0.3974603, 0.20965295, 0.80379957]","[0.9996054, -0.00017237716, -0.0061986432, -0....","[0.43440193, 0.20762655, 0.80425847]",-0.092784,0.238087,0.058688,-1,-1
2,100000000,2,"[0.018771406, -0.47597754, -0.008583693, -2.77...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.001...","[0.3566572, -0.25636595, 1.0521638, 0.00047673...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.001...",approach,"[0.39746037, 0.20965214, 0.8037997]","[0.9996062, -0.00017243318, -0.006196845, -0.0...","[0.4344021, 0.20762771, 0.8042584]",-0.092784,0.238087,0.058635,-1,-1


### Compatibility with the lab's loaders
`Trajectory` reads `timestamp` as ns, requires equal spacing, and loads `<label>.mp4` for each label
in `config.FEED_IMAGE_LABELS_TO_USE`. To train the Cosmos-latent WM on sim data, point it at an
episode folder and set
`FEED_IMAGE_LABELS_TO_USE = ["right_gripper_front", ...]`, `STATE_OBS_DIM = 44`, `ACTION_DIM = 16`.

In [5]:
ts = df["timestamp"].values.astype(np.float64) * 1e-9
assert np.allclose(np.diff(ts), np.diff(ts)[0], atol=1e-6), "timestamps must be equally spaced"
print(f"dt = {np.diff(ts)[0]:.3f}s equally spaced ✓; files:", sorted(p.name for p in pathlib.Path(idx.path.iloc[0]).iterdir()))

dt = 0.050s equally spaced ✓; files: ['annotations.json', 'episode.parquet', 'feature_report.json', 'left_gripper_front.mp4', 'left_gripper_rear.mp4', 'right_gripper_front.mp4', 'right_gripper_rear.mp4', 'sim_states.npz']


## Counterfactual branching from a saved state (the RQ1 "oracle" mechanic)
Rebuild an episode up to a decision point (the start of `insert`), then execute **K candidate
action chunks** from the identical state. The chunks are the recorded chunk with a lateral offset
of growing size in a random direction. Then the base policy (the expert, retries off) takes over,
and we record whether each candidate ends latched. This is the ground truth that imagined-rollout
scores (WM σ, success-vs-failure WM, …) get ranked against.

In [6]:
from plugsim.env import PlugEnv
from plugsim.datagen import EpisodeSpec
from plugsim.tools import expert_at, chunk_then_expert, clone_expert, restore_expert
nom = idx[(idx.category == "nominal") & (idx.outcome == "success")].iloc[0]
E = load_episode(nom.path)
spec = EpisodeSpec(**E["annotations"]["spec"])
k = int(np.argmax(E["df"]["priv.phase"].values == "insert"))
env = PlugEnv(render=False)
ex = expert_at(env, spec, k)
assert np.abs(env.observe() - E["df"]["observation.state"][k]).max() < 1e-5  # identical to the recording
snap, exs = env.get_state(), clone_expert(ex)
H, K, rng = 30, 16, np.random.default_rng(0)
arm = ["right", "left"].index(E["annotations"]["active_arm"])
results = []
for i in range(K):
    env.set_state(snap); restore_expert(ex, exs); ex.max_retries = 0
    chunk = E["actions64"][k : k + H].copy()
    r = 4e-3 * i / (K - 1); ang = rng.uniform(0, 2 * np.pi)
    chunk[0, 8 * arm + 1 : 8 * arm + 3] += r * np.array([np.cos(ang), np.sin(ang)])  # lateral jump, kept for the chunk
    ok = chunk_then_expert(env, ex, chunk)
    results.append(dict(candidate=i, offset_mm=round(r * 1e3, 2), direction_deg=round(np.degrees(ang)), success=ok,
                        max_depth_mm=round(env.privileged()["depth"] * 1e3, 1)))
R = pd.DataFrame(results); print(f"oracle success rate over candidates: {R.success.mean():.0%}"); R

oracle success rate over candidates: 56%


,candidate,offset_mm,direction_deg,success,max_depth_mm
0,0,0.00,229,True,17.3
1,1,0.27,97,True,17.5
2,2,0.53,15,True,17.4
3,3,0.80,6,True,17.4
4,4,1.07,293,True,16.9
5,5,1.33,329,True,17.4
6,6,1.60,218,True,16.6
7,7,1.87,263,True,17.0
8,8,2.13,196,False,-0.7
9,9,2.40,337,True,15.9


## Scaling up on an H100 node

```bash
# one node (e.g. 2x48-core host with 8xH100): physics on CPU cores, rendering via EGL
cd RobotPluggingSim && pip install -e .
MUJOCO_GL=egl python scripts/generate_dataset.py --out /scratch/plugsim/v0 --n 20000 --workers 90

# shard over several nodes / SLURM array: disjoint seed ranges, then concat the index files
MUJOCO_GL=egl python scripts/generate_dataset.py --out /scratch/plugsim/v0 --n 20000 --seed0 $((SLURM_ARRAY_TASK_ID*20000)) --workers 90

# variants without code changes (EnvCfg overrides), e.g. tighter jack + smaller frames + all cameras
python scripts/generate_dataset.py --out /scratch/plugsim/tight --n 5000 \
    --cfg '{"port.clearance": 0.0005, "camera.width": 256, "camera.height": 144}' \
    --cameras right_gripper_front,right_gripper_rear,left_gripper_front,left_gripper_rear,head_front,chest_left,chest_right
```

Physics-only runs (`--cameras none`) are about 4× faster and still give state, actions, labels and
snapshots. Frames can be rendered later from `sim_states.npz`. The GPUs are mainly for what comes
next: training policies and the Cosmos-latent world model on these episodes, and batched
counterfactual scoring.

In [7]:
per_ep_worker_s = wall * WORKERS / N
for cores in (8, 48, 96):
    print(f"{cores:3d} workers ≈ {cores / per_ep_worker_s:5.1f} episodes/s ≈ {3600 * cores / per_ep_worker_s / 1000:5.1f}k episodes/hour "
          f"≈ {3600 * cores / per_ep_worker_s * size_mb / N / 1000:5.1f} GB/hour  (extrapolated from this machine)")

  8 workers ≈   0.4 episodes/s ≈   1.4k episodes/hour ≈  13.1 GB/hour  (extrapolated from this machine)
 48 workers ≈   2.3 episodes/s ≈   8.4k episodes/hour ≈  78.5 GB/hour  (extrapolated from this machine)
 96 workers ≈   4.7 episodes/s ≈  16.8k episodes/hour ≈ 157.1 GB/hour  (extrapolated from this machine)
